# TRUEWATCH Phase 1 build on Kaggle

Builds the unified detection corpus (IDD Detection + Teledyne FLIR ADAS v2 + LLVIP, plus the elevated-view and
thermal sources HIT-UAV, AAU-PD-T, BIRDSAI and VisDrone) end to end:
`00_fetch` → `01..03` converters → `04..09` → `11_stats` → `10_validate`.

**Attach these seven datasets as inputs** (Add Input → Datasets), nothing else needs uploading:

| source | Kaggle dataset |
|---|---|
| IDD Detection (third-party mirror) | `vinayak21574/idd-detection` |
| Teledyne FLIR ADAS v2 (third-party mirror) | `samdazel/teledyne-flir-adas-thermal-dataset-v2` |
| LLVIP (third-party mirror) | `afradhossain/llvip-dataset` |
| HIT-UAV (third-party mirror) | `pandrii000/hituav-a-highaltitude-infrared-thermal-dataset` |
| AAU-PD-T (third-party mirror) | `noorulhuda90/aaupdt` |
| BIRDSAI (third-party mirror) | `manitagarwal/birdsai` |
| VisDrone2019-DET (third-party mirror) | `kushagrapandya/visdrone-dataset` |

Settings: internet **on** (to clone the repository), accelerator **none** (the build is CPU only; 4 cores).
The inputs are read-only and are never copied. Intermediate state lives in `/tmp`; only the finished
dataset is built in `/tmp/truewatch_ds/` and saved to `/kaggle/working/truewatch_ds_shards/` as a few tar shards
(Kaggle does not keep ~170k loose output files; target ≤ 15 GB of the 20 GB output limit).

The output directory is self-contained: `data.yaml` (relative `path`), `images/`, `labels/`,
`manifest.tsv`, `index/split.jsonl`, `manifests/` (hard set, splits, negatives, cart gate),
`reports/` (stats, charts, validation table, logs) and `reports/spotcheck/` (100 rendered images for
the manual gate G19). Save the notebook version, then attach its output to the training notebook.

In [ ]:
# ---- parameters ------------------------------------------------------------------------------
REPO_URL = "https://github.com/0XSreekar/True-Watch-AI.git"
BRANCH = "fix/phase1-2-complete"
OUT = "/tmp/truewatch_ds"                 # built on local disk; packed into SHARDS below
SHARDS = "/kaggle/working/truewatch_ds_shards"  # the saved output: a few tar shards, not ~170k loose files
SHARD_GB = 2.0                            # target uncompressed size per shard (JPEGs do not compress)
PROC = "/tmp/truewatch_processed"         # intermediate state; not saved, not counted
CODE = "/tmp/True-Watch-AI"               # repository clone; not saved
INPUT_BASE = "/kaggle/input"
SEED = 42
WORKERS = 4                               # Kaggle CPU sessions have 4 cores
INSTALL_PINNED = True                     # install datasets/requirements.txt exactly
LIMIT = None                              # e.g. 200 for a quick rehearsal on a subset

In [ ]:
import os, subprocess, sys, time, json, shutil
from pathlib import Path

def sh(cmd, cwd=None, check=True):
    print(f"$ {cmd}", flush=True)
    result = subprocess.run(cmd, shell=True, cwd=cwd)
    if check and result.returncode != 0:
        raise RuntimeError(f"command failed ({result.returncode}): {cmd}")
    return result.returncode

if Path(CODE).exists():
    shutil.rmtree(CODE)
sh(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} {CODE}")
sh(f"git -C {CODE} log -1 --format='%H %s'")
if INSTALL_PINNED:
    sh(f"{sys.executable} -m pip install -q -r {CODE}/datasets/requirements.txt")
sh(f"{sys.executable} -c 'import cv2, numpy, yaml; print(cv2.__version__, numpy.__version__)'")

## Locate the three inputs

Kaggle mounts inputs at `/kaggle/input/<slug>/...` or `/kaggle/input/datasets/<owner>/<slug>/...`
depending on how they were attached, so each source root is found by searching for a marker file
rather than by a hard-coded path.

In [ ]:
MARKERS = {
    # source: a file that must exist inside that source's root
    "flir":  "images_thermal_train/coco.json",
    "idd":   "train.txt",
    "llvip": "Annotations/010001.xml",
    "hituav": "labels/train",
    "aaupdt": "Annotation_Format.txt",
    "birdsai": "TestReal",
    "visdrone": "VisDrone.yaml",
}
EXTRA_CHECK = {
    "flir":  lambda root: (root / "images_rgb_train" / "coco.json").exists(),
    "idd":   lambda root: (root / "Annotations").is_dir() and (root / "JPEGImages").is_dir(),
    "llvip": lambda root: (root / "infrared").is_dir() and (root / "visible").is_dir(),
    "hituav": lambda root: (root / "images" / "train").is_dir() and (root / "dataset.yaml").exists(),
    "aaupdt": lambda root: (root / "Train").is_dir() and (root / "Test").is_dir(),
    "birdsai": lambda root: (root / "TestReal").is_dir(),
    "visdrone": lambda root: any(root.glob("VisDrone2019-DET-train*")),
}

def discover(base, max_depth=6):
    found = {}
    base = Path(base)
    for directory, dirs, files in os.walk(base, followlinks=True):
        depth = len(Path(directory).relative_to(base).parts)
        if depth >= max_depth:
            dirs[:] = []
        dirs.sort()
        here = Path(directory)
        for name, marker in MARKERS.items():
            if name in found:
                continue
            first = marker.split("/")[0]
            if first in dirs or first in files:
                if (here / marker).exists() and EXTRA_CHECK[name](here):
                    found[name] = here
        if len(found) == len(MARKERS):
            break
    return found

ROOTS = discover(INPUT_BASE)
for name in MARKERS:
    print(f"{name:<6} -> {ROOTS.get(name, 'NOT FOUND')}")
missing = [n for n in MARKERS if n not in ROOTS]
if missing:
    raise SystemExit(f"attach the missing input(s): {missing}")

In [ ]:
os.makedirs(OUT, exist_ok=True)
os.makedirs(PROC, exist_ok=True)
LOGS = Path(OUT) / "reports" / "logs"
LOGS.mkdir(parents=True, exist_ok=True)
os.environ["TRUEWATCH_MANIFEST_DIR"] = f"{OUT}/manifests"
os.environ["TRUEWATCH_REPORT_DIR"] = f"{OUT}/reports"
os.environ["TRUEWATCH_WORKERS"] = str(WORKERS)
os.environ["PYTHONUNBUFFERED"] = "1"

COMMON = f"--processed {PROC} --seed {SEED} --workers {WORKERS}"
SUBSET = f" --limit {LIMIT} --subset-ok" if LIMIT else ""
TIMINGS = {}

def step(script, extra="", fatal=True):
    # Run one pipeline script, stream its log, keep a copy under reports/logs/.
    cmd = f"{sys.executable} datasets/scripts/{script} {COMMON} {extra}".strip()
    print(f"\n==== {script}", flush=True)
    started = time.time()
    with open(LOGS / f"{Path(script).stem}.log", "w") as log:
        proc = subprocess.Popen(cmd, shell=True, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line)
        code = proc.wait()
    TIMINGS[script] = (round((time.time() - started) / 60.0, 1), code)
    print(f"==== {script} exit={code} in {TIMINGS[script][0]} min", flush=True)
    if code != 0 and fatal:
        raise RuntimeError(f"{script} failed with exit {code}; see {LOGS / (Path(script).stem + '.log')}")
    return code

## Run the pipeline

Estimated wall time on a 4-core Kaggle CPU session: converters 3-6 min, `04` 3-5 min, `05` (pHash)
8-15 min, `06`-`08` 3-6 min, `09` (encode ~90k JPEGs) 20-40 min, `11` + `10` 3-6 min: about 45-80 min
in total, far inside the 12 h limit. `07` and `10` are allowed to fail without stopping the notebook:
their failures are gates, and the gate table below reports them.

In [ ]:
roots = " ".join(f"--root {k}={v}" for k, v in ROOTS.items())
step("00_fetch.py", roots)
step("01_convert_idd.py", SUBSET)
step("02_convert_flir.py", SUBSET)
step("03_convert_llvip.py", f" --limit {LIMIT}" if LIMIT else "")
for extra in ("hituav", "aaupdt", "birdsai", "visdrone"):
    step("12_convert_extra.py", f"--dataset {extra}" + (f" --limit {LIMIT}" if LIMIT else ""))
step("04_ir_to_3ch.py", "--max-failures 20")
step("05_dedupe.py")
step("06_split.py")
step("07_negatives.py", fatal=False)
step("08_tile_farfield.py")
# Seven sources no longer fit the 20 GB output at quality 92 / 1280 px, so frames are capped at
# 1152 px and written at quality 90 (training runs at 640 px; far-field tiles are cut before the cap).
step("09_build_yolo_ds.py", f"--out {OUT} --quality 90 --long-side 1152")
step("11_stats.py", f"--dataset {OUT}")

In [ ]:
# The gate. Exit 1 is expected until a human writes reports/spotcheck/VERDICT.txt (G19).
# G17 (plates) is out of scope for this detection build and is reported as SKIP, never PASS.
step("10_validate.py", f"--dataset {OUT} --skip-plates", fatal=False)

# Pipeline-side reports travel with the build for provenance.
dest = Path(OUT) / "reports" / "pipeline"
dest.mkdir(parents=True, exist_ok=True)
for path in sorted((Path(PROC) / "reports").glob("*.json")):
    shutil.copy2(path, dest / path.name)
if (Path(PROC) / "fetch_manifest.json").exists():
    shutil.copy2(Path(PROC) / "fetch_manifest.json", dest / "fetch_manifest.json")

## Summary

In [ ]:
stats = json.loads((Path(OUT) / "reports" / "stats.json").read_text())
validation = json.loads((Path(OUT) / "reports" / "validation.json").read_text())
names = list(stats["class_histogram"].keys())

print("images per split x modality, instances per class")
print(f"  {'split':<6} {'modality':<8} {'images':>8} " + " ".join(f"{n[:11]:>11}" for n in names))
for split, modalities in stats["by_split_modality"].items():
    for modality, count in modalities.items():
        row = stats["instances_by_split_modality_class"].get(split, {}).get(modality, {})
        print(f"  {split:<6} {modality:<8} {count:>8} " + " ".join(f"{row.get(n, 0):>11}" for n in names))
print()
print("visible share per split :", stats["visible_share"], "(gate G8: 0.55-0.70)")
print("negatives share, train  :", stats["negatives_share_train"], "(gate G9: 0.08-0.12)")
print("untiled proportions     :", stats["untiled_split_proportions"], "(gate G14: 0.70/0.15/0.15 +/- 0.03)")
print("tiles                   :", stats["tiles"])
print("class share             :", stats["class_share"])
print("cart gate               :", stats["cart_gate"].get("decision"), "-", stats["cart_gate"].get("reason"))
print()
print(f"{'gate':<5} {'result':<6} {'check':<32} observed")
for g in validation["gates"]:
    print(f"{g['id']:<5} {g['result']:<6} {g['check']:<32} {g['observed']}")
print("failed :", validation["failed"])
print("skipped:", validation["skipped"])
print()
total = sum(p.stat().st_size for p in Path(OUT).rglob("*") if p.is_file())
print(f"output size: {total / 1024**3:.2f} GB in {OUT} (limit 20 GB)")
print("step timings:")
for script, (minutes, code) in TIMINGS.items():
    print(f"  {script:<22} {minutes:>6} min  exit={code}")

In [ ]:
# Kaggle keeps a notebook's output only as a modest number of files: ~170k loose images and labels
# are silently dropped when the version is saved. The finished dataset is therefore packed into a
# few tar shards (plain tar: JPEGs do not compress) that the training notebook unpacks onto local
# disk. The 100 spot-check renders and the charts are also copied loose so they can be viewed.
import hashlib, tarfile

shard_dir = Path(SHARDS)
if shard_dir.exists():
    shutil.rmtree(shard_dir)
shard_dir.mkdir(parents=True)
root = Path(OUT)
meta_files, image_files = [], []
for path in sorted(p for p in root.rglob("*") if p.is_file()):
    rel = path.relative_to(root.parent)
    (image_files if rel.parts[1] == "images" else meta_files).append((path, rel))

def write_shard(name, members):
    target = shard_dir / name
    with tarfile.open(target, "w") as tar:
        for path, rel in members:
            tar.add(path, arcname=str(rel), recursive=False)
    return target

shards = [write_shard("truewatch_ds_meta.tar", meta_files)]
limit = SHARD_GB * 1024**3
batch, size, k = [], 0, 0
for path, rel in image_files:
    batch.append((path, rel))
    size += path.stat().st_size
    if size >= limit:
        shards.append(write_shard(f"truewatch_ds_images_{k:02d}.tar", batch)); k += 1
        batch, size = [], 0
if batch:
    shards.append(write_shard(f"truewatch_ds_images_{k:02d}.tar", batch))

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 22), b""):
            h.update(chunk)
    return h.hexdigest()

index = {"format": "truewatch_ds tar shards v1", "extract_to_parent_of": "truewatch_ds",
         "files": len(meta_files) + len(image_files),
         "shards": [{"name": s.name, "bytes": s.stat().st_size, "sha256": sha256(s)} for s in shards]}
(shard_dir / "SHARDS.json").write_text(json.dumps(index, indent=2))
loose = Path(SHARDS) / "view"
for sub in ["reports/spotcheck"]:
    if (root / sub).is_dir():
        shutil.copytree(root / sub, loose / sub)
for chart in (root / "reports").glob("*.png"):
    (loose / "reports").mkdir(parents=True, exist_ok=True)
    shutil.copy2(chart, loose / "reports" / chart.name)
for name in ["data.yaml"]:
    shutil.copy2(root / name, loose / name)
total = sum(s.stat().st_size for s in shards)
print(f"{len(shards)} shards, {total / 1024**3:.2f} GB, {index['files']} files packed; loose view files: "
      f"{sum(1 for p in loose.rglob('*') if p.is_file())}")


## After the run

1. Open `reports/spotcheck/` (100 final training images with boxes drawn; `index.tsv` maps each to its
   dataset file). Look at them. Write `PASS` or `FAIL` plus one line of what you saw to
   `reports/spotcheck/VERDICT.txt` and re-run `10_validate.py --dataset <this output>` wherever the
   output is mounted. G19 cannot pass without a human.
2. Class 4 (`cart`) is withdrawn unless `manifests/vehicle_fallback.csv` holds at least 300 rows a
   human marked `cart` (DATASET_SPEC 1.5). The id stays reserved in `data.yaml`.
3. Save a version of this notebook and add its output as an input to the training notebook. The output is
   `truewatch_ds_shards/` (tar shards plus `SHARDS.json`); the training notebook verifies and unpacks them itself. The
   training scripts find `data.yaml` under `/kaggle/input` by themselves; the hard set is at
   `manifests/hard_set.txt` and the split index at `index/split.jsonl`.